# NLP Using Transformers: Question-Answering Chatbot
Dataset: [Project Gutenberg's Frankenstien; or, The Modern Promethus by Mary Shelley](https://www.gutenberg.org/ebooks/84)

This file focuses on using the **transformer model** to perform the NLP task of **developing a question-answering system (chatbot) on the text** of an English language book from Project Gutenberg, using its text format. We use **pre-trained** transformer models from [Hugging Face](https://huggingface.co/models): sentence embeddings to find the passages of the book most relevant to a question, and a [DistilBERT](https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad) question-answering model to extract the answer from them. The chatbot is then evaluated using appropriate evaluation metrics.

### 0. Installing Libraries
Installing and importing the necessary libraries for the project.

In [4]:
## 0.0 Installing packages and importing libraries
%pip install -q "transformers[sentencepiece]<5" torch


[notice] A new release of pip is available: 23.3.2 -> 26.3
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
## 0.1. Importing libraries
import requests
import string
import torch
import torch.nn.functional as F
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel
)

# 0.1.1. Selecting the device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():     # Apple Silicon
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
    
print('torch version: {}'.format(torch.__version__))
print('Device: {}'.format(DEVICE))

/Users/dylanramdhan/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.2.2
Device: mps


### 1. Project and Dataset Selection
In Project Gutenberg, we've selected the English language book [**Frankenstein; or, The Modern Prometheus** by Mary Shelley](https://www.gutenberg.org/ebooks/84), and use its **text format** for this assignment.

In [6]:
## 1.0. Downloading the book's text format from Project Gutenberg
book_url = 'https://www.gutenberg.org/cache/epub/84/pg84.txt'

response = requests.get(book_url)
raw_text = response.content.decode('utf-8')

print('Characters downloaded: {:,}'.format(len(raw_text)))

Characters downloaded: 446,582


In [7]:
## 1.2. Removing the Project Gutenberg header and footer (License text; Not part of the book)
start = raw_text.find('*** START OF') # header ends
start = raw_text.find('\n', start) # skips next line after header
end = raw_text.find('*** END OF')

book_text = raw_text[start : end].strip() # keeping in between start/end, trimming blank space

# 1.2.1. Removing the table of contents (Story starts at the second "Letter 1")
FirstLetter = book_text.find('Letter 1')                 # inside table of contents
StoryStart = book_text.find('Letter 1', FirstLetter + 1) # where story actually begins
book_text = book_text[StoryStart:]

print('Characters in book: {:,}'.format(len(book_text)))
print('Words in book: {:,}'.format(len(book_text.split())))
print()
print(book_text[:500]) # first 500 characters; checking if header is gone

Characters in book: 426,233
Words in book: 74,975

Letter 1

_To Mrs. Saville, England._


St. Petersburgh, Dec. 11th, 17—.


You will rejoice to hear that no disaster has accompanied the
commencement of an enterprise which you have regarded with such evil
forebodings. I arrived here yesterday, and my first task is to assure
my dear sister of my welfare and increasing confidence in the success
of my undertaking.

I am already far north of London, and as I walk in the streets of
Petersburgh, I feel a cold northern breeze play upon 


### 2. Pre-Processing: Splitting the Book into Passages
The question-answering model can only read a limited amount of text at once (up to 384 tokens, including the question), so the full book cannot be given to it directly. We split the book into short  **passages** of a fixed number of words. Later, the chatbot searches these passages for the one most relevant to a question and gives only those to the model.

In [8]:
## 2.1. Splitting the book into passages of CHUNK_WORDS words
CHUNK_WORDS = 150

words = book_text.split()

passages = []
for i in range(0, len(words), CHUNK_WORDS):
    passages_words = words[i:i + CHUNK_WORDS]
    passages.append(' '.join(passages_words))
    
print('Total words: {:,}'.format(len(words)))
print('Number of passages: {:,}'.format(len(passages)))

Total words: 74,975
Number of passages: 500


In [9]:
## 2.2. Looking at an example passage
print('Passage 100:')
print(passages[100])

Passage 100:
filled my heart. Unable to endure the aspect of the being I had created, I rushed out of the room and continued a long time traversing my bed-chamber, unable to compose my mind to sleep. At length lassitude succeeded to the tumult I had before endured, and I threw myself on the bed in my clothes, endeavouring to seek a few moments of forgetfulness. But it was in vain; I slept, indeed, but I was disturbed by the wildest dreams. I thought I saw Elizabeth, in the bloom of health, walking in the streets of Ingolstadt. Delighted and surprised, I embraced her, but as I imprinted the first kiss on her lips, they became livid with the hue of death; her features appeared to change, and I thought that I held the corpse of my dead mother in my arms; a shroud enveloped her form, and I saw the grave-worms


### 3. Retrieval: Finding the Passages Related to a Question
To find which passages of the book a question is about, we use a pre-trained **sentence-embedding** transformer, [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). It converts passage into a vector of 384 numbers that represents its meaning. A question is converted the same way, and the passages whose vectors are most similar to the question's (by **consine similarity**) are the ones most likely to contain the answer. This is the retrieval step of **retrieval-augmented generation (RAG)**

In [10]:
## 3.1. Loading the sentence-embedding vectors
EMB_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
emb_tok = AutoTokenizer.from_pretrained(EMB_NAME)
emb_model = AutoModel.from_pretrained(EMB_NAME).to(DEVICE).eval()

# 3.1.1. Turning a list of texts into embedding vectors
def EmbedTexts(texts):
    batch = emb_tok(texts, padding = True, truncation = True, return_tensors = 'pt').to(DEVICE)
    with torch.no_grad():
        token_emb = emb_model(**batch).last_hidden_state
    mask = batch['attention_mask'].unsqueeze(-1).float()
    sent_emb = (token_emb * mask).sum(1) / mask.sum(1)
    
    return F.normalize(sent_emb, p = 2, dim = 1)

In [11]:
## 3.2. Embedding every passage of the book
BATCH_SIZE = 32 # passages embedded at a time

passage_vectors = []
for i in range(0, len(passages), BATCH_SIZE):
    batch_passages = passages[i : i+BATCH_SIZE]
    passage_vectors.append(EmbedTexts(batch_passages))
    
passage_vectors = torch.cat(passage_vectors) # join batches into one table

print('Passage vetors shape: {}'.format(tuple(passage_vectors.shape)))

Passage vetors shape: (500, 384)


In [12]:
## 3.3. Searching the book for the passages most related to a Question
TOP_K = 3

def FindPassages(question, top_k):
    question_vector = EmbedTexts([question])
    scores = (question_vector @ passage_vectors.T)[0] # cosine similarity to every passage
    best = torch.topk(scores, top_k) # highest scores for top_k
    results = []
    
    for score, index in zip(best.values, best.indices):
        results.append((index.item(), score.item())) # (passage number, similarity score)
    return results

In [ ]:
## 3.4. Testing the search with an sample question
# Assumption: 
#   - Observation: with "Victor Frankenstein" in the question, the search ranked passages that mention
#       the name "Frankenstein" highest, and missed the passage containing the answer (Ingolstadt).
#       Using only "Victor" let the word "university" carry more weight, and the answer passage was found.
question = 'Where did Victor go to university?'

# for index, score in FindPassages(question, TOP_K):
#     print('Passage {} (similarity {:.2f}):'.format(index, score))
for index, score in FindPassages(question, TOP_K):
    print('Passage {}: contains Ingolstadt = {}'.format(index, 'Ingolstadt' in passages[index]))
    print(passages[index][:300] + '...')
    print()

Passage 122: contains Ingolstadt = False
M. Duvillard, the rich banker, last autumn. Your favourite schoolfellow, Louis Manoir, has suffered several misfortunes since the departure of Clerval from Geneva. But he has already recovered his spirits, and is reported to be on the point of marrying a lively pretty Frenchwoman, Madame Tavernier. ...

Passage 126: contains Ingolstadt = False
if you please; but it is nevertheless true. A youngster who, but a few years ago, believed in Cornelius Agrippa as firmly as in the gospel, has now set himself at the head of the university; and if he is not soon pulled down, we shall all be out of countenance.—Ay, ay,” continued he, observing my fa...

Passage 133: contains Ingolstadt = False
to you the horrible tidings. “William is dead!—that sweet child, whose smiles delighted and warmed my heart, who was so gentle, yet so gay! Victor, he is murdered! “I will not attempt to console you; but will simply relate the circumstances of the transaction. “Last 

### 4. Question Answering: Extracting the Answer
To answer a question, we use a pre-trained **DistilBERT** transformer fine-tuned on the [SQuAD](https://huggingface.co/datasets/squad) question-answering dataset ([distilbert-base-cased-distilled-squad](https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad)). 

Once we are give a **question** and a **passage**, it predicts where the answer **starts** and **ends** in the passage, and returns that span along with a confidence score. Our chatbot gives it the top passages found in Section 3, and keeps the answer with the highest confidence.

In [ ]:
## 4.1. Loading the question-answering model
QA_NAME = 'distilbert/distilbert-base-cased-distilled-squad'
qa_pipeline = pipeline('question-answering', model = QA_NAME, device=DEVICE)

Device set to use mps


In [ ]:
## 4.2. The chatbot: Search the book, then extract the answer
def AnswerQuestion(question, top_k):
    best_answer = None
    for index, similarity in FindPassages(question, top_k):
        result = qa_pipeline(question = question, context = passages[index])
        
        if best_answer is None or result['score'] > best_answer['score']:
            best_answer = result
            best_answer['passage'] = index
    return best_answer

# 4.2.1. Testing the chatbot
question = 'Where did Victor go to university?'
answer = AnswerQuestion(question, TOP_K)

print('Question: {}'.format(question))
print('Answer: {}'.format(answer['answer']))
print('Confideence: {:.2f} (from passage {})'.format(answer['score'], answer['passage']))

Question: Where did Victor go to university?
Answer: Plainpalais
Confideence: 0.43 (from passage 133)


### 5. Evaluation
To evaluate the chatbot, we wrote a set of factual questions about the book whose answers appear word-for-word in the text. The chatbot answers each question, and we compare its answer with the correct answer using the two standard metrics for extractive question answering on [SQuAD](https://huggingface.co/datasets/squad), the dataset the DistilBERT model was trained on:
    
- **Exact Match (EM):** It would be 1 if the predicted answer matches the correct answer exactly (after lowercasing and removing punctuation and the words "a", "an", "the"), otherwise 0.
    
- **F1 Score:** measures the word overlap between the predicted and correct answers, giving partial credit for answers that are close.

In [ ]:
## 5.1. Evaluation Questions with known answers from the book
evaluate_questions = [
    ('Where did Victor go to university?', 'Ingolstadt'),
    ('To whom does Robert Walton write his letters?', 'Mrs. Saville'),
    ('What city is Victor\'s family from?', 'Geneva'),
    ('Which professor inspired Victor to study chemistry?', 'Waldman'),
    ('Which professor dismissed the alchemists Victor had read?', 'Krempe'),
    ('Which alchemist did Victor read as a boy?', 'Cornelius Agrippa'),
    ('Who is Victor\'s closest friend?', 'Clerval'),
    ('What is the name of Victor\'s youngest brother?', 'William'),
    ('Who was accused of murdering William?', 'Justine'),
    ('Whom does Victor marry?', 'Elizabeth'),
    ('What is the name of Victor\'s father?', 'Alphonse'),
    ('What is the name of the family the creature watches from the hovel?', 'De Lacey'),
]

# 5.1.1. Checking that every answer appears in the book
for question, answer in evaluate_questions:
    # edge cases for any given error
    if answer in book_text:
        print('FOUND:     {}'.format(answer))
   
   # questions to fix
    else: 
        print('NOT FOUND: {} <--- fix/remove this question'.format(answer))
        
        
print('Number of evaluation questions: {}'.format(len(evaluate_questions)))

FOUND:     Ingolstadt
FOUND:     Mrs. Saville
FOUND:     Geneva
FOUND:     Waldman
FOUND:     Krempe
FOUND:     Cornelius Agrippa
FOUND:     Clerval
FOUND:     William
FOUND:     Justine
FOUND:     Elizabeth
FOUND:     Alphonse
FOUND:     De Lacey
Number of evaluation questions: 12


In [ ]:
## 5.2. Exact Match and F1 Score
# (SQuAD evaluation: compare answers after lowercasing, removing punctuation, and any articles)
def NormalizeAnswer(text):
    text = text.lower()
    for mark in string.punctuation:
        text = text.replace(mark, '')
    
    words = []
    for word in text.split():
        if word not in ['a', 'an', 'the']:
            words.append(word)
    return words


def ExactMatch(prediction, truth):
    if NormalizeAnswer(prediction) == NormalizeAnswer(truth):
        return 1
    else:
        return 0
    

def F1Score(prediction, truth):
    pred_words = NormalizeAnswer(prediction)
    true_words = NormalizeAnswer(truth)
    
    common = 0
    remaining = list(true_words)
    for word in pred_words:
        if word in remaining:
            common = common + 1
            remaining.remove(word)
            
    if common == 0:
        return 0.0
    precision = common / len(pred_words)
    recall = common / len(true_words)
    return 2 * precision * recall / (precision + recall)


# 5.2.1. Quick check of the two metrics
print(ExactMatch('Ingolstadt', 'Ingolstadt'))                                  
print(ExactMatch('the university of Ingolstadt', 'Ingolstadt'))                 
print('{:.2f}'.format(F1Score('the university of Ingolstadt', 'Ingolstadt'))) 

1
0
0.50


In [21]:
## 5.3. Evaluating the chatbot on all evaluation questions
def EvaluateChatbot(top_k, show_details):
    total_em = 0
    total_f1 = 0.0
    
    for question, true_answer in evaluate_questions:
        predicted = AnswerQuestion(question, top_k)['answer']
        em = ExactMatch(predicted, true_answer)
        f1 = F1Score(predicted, true_answer)
        
        total_em = total_em + em
        total_f1 = total_f1 + f1        
        
        if show_details:
            print('EM = {}, F1 = {:.2f} | Q: {}'.format(em, f1, question))
            print(' Predicted: {} | Correct: {}'.format(predicted, true_answer))
        
        
    average_em = total_em / len(evaluate_questions)
    average_f1 = total_f1 / len(evaluate_questions)
    return average_em, average_f1


# 5.3.1. Scores with the current settings
average_em, average_f1 = EvaluateChatbot(TOP_K, show_details = True)
print()
print('CHUNK WORDS = {}, TOP_K = {}'.format(CHUNK_WORDS, TOP_K))
print('Exact Match: {:.1%}'.format(average_em))
print('F1 Score:    {:.1%}'.format(average_f1))

EM = 0, F1 = 0.00 | Q: Where did Victor go to university?
 Predicted: Plainpalais | Correct: Ingolstadt
EM = 0, F1 = 0.80 | Q: To whom does Robert Walton write his letters?
 Predicted: Mrs. Saville, England | Correct: Mrs. Saville
EM = 0, F1 = 0.00 | Q: What city is Victor's family from?
 Predicted: Plainpalais | Correct: Geneva
EM = 0, F1 = 0.00 | Q: Which professor inspired Victor to study chemistry?
 Predicted: M. Krempe | Correct: Waldman
EM = 0, F1 = 0.67 | Q: Which professor dismissed the alchemists Victor had read?
 Predicted: M. Krempe | Correct: Krempe
EM = 0, F1 = 0.00 | Q: Which alchemist did Victor read as a boy?
 Predicted: Alphonse Frankenstein | Correct: Cornelius Agrippa
EM = 0, F1 = 0.00 | Q: Who is Victor's closest friend?
 Predicted: Louisa Biron | Correct: Clerval
EM = 0, F1 = 0.00 | Q: What is the name of Victor's youngest brother?
 Predicted: Alphonse Frankenstein | Correct: William
EM = 0, F1 = 0.00 | Q: Who was accused of murdering William?
 Predicted: my little

### 6. Hyperparameters Tuning
The chatbot's accuracy depends on how the book is split into passages and how many passages are searched. We tune the 3 hyperparameters, scoring each combination on the same evaluation questions:
- **CHUNK_WORDS**: the number of words per passage (100, 150, 200)
- **OVERLAP**: the number of words shared between neighboring passages (0, 50), so that a sentence cut at the edge of one passage still appears whole in the next
- **TOP_K**: the number of passages given to the question-answering model (3, 5, 10)

In [ ]:
## 6.1. Rebuilding the passages and their vectors for a given passsage size and overlap
def RebuildingPassages(chunk_words, overlap):
    global passages, passage_vectors
    
    words = book_text.split()
    step = chunk_words - overlap
    passages = []
    
    for i in range (0, len(words), step):
        passages.append(' '.join(words[i : i + chunk_words]))
    
    passage_vectors = []
    for i in range(0, len(passages), BATCH_SIZE):
        passage_vectors.append(EmbedTexts(passages[i : i + BATCH_SIZE]))
    
    passage_vectors = torch.cat(passage_vectors)

In [25]:
## 6.2. Testing every combination of settings
chunk_options = [100, 150, 200]
overlap_options = [0, 50]
top_k_options = [3, 5, 10]

tuning_results = []
for chunk_words in chunk_options:
    for overlap in overlap_options:
        RebuildingPassages(chunk_words, overlap)
        
        for top_k in top_k_options:
            em, f1 = EvaluateChatbot(top_k, show_details = False) # Not wanting to show all details for output
            tuning_results.append((chunk_words, overlap, top_k, em, f1))
            
            print('CHUNK_WORDS = {:3}, OVERLAP = {:2}, TOP_K = {:2} | EM = {:.1%}, F1 = {:.1%}'.format(chunk_words, overlap, top_k, em, f1))

CHUNK_WORDS = 100, OVERLAP =  0, TOP_K =  3 | EM = 16.7%, F1 = 28.9%
CHUNK_WORDS = 100, OVERLAP =  0, TOP_K =  5 | EM = 16.7%, F1 = 23.3%
CHUNK_WORDS = 100, OVERLAP =  0, TOP_K = 10 | EM = 16.7%, F1 = 22.2%
CHUNK_WORDS = 100, OVERLAP = 50, TOP_K =  3 | EM = 8.3%, F1 = 26.1%
CHUNK_WORDS = 100, OVERLAP = 50, TOP_K =  5 | EM = 16.7%, F1 = 34.4%
CHUNK_WORDS = 100, OVERLAP = 50, TOP_K = 10 | EM = 8.3%, F1 = 8.3%
CHUNK_WORDS = 150, OVERLAP =  0, TOP_K =  3 | EM = 0.0%, F1 = 17.8%
CHUNK_WORDS = 150, OVERLAP =  0, TOP_K =  5 | EM = 8.3%, F1 = 25.0%
CHUNK_WORDS = 150, OVERLAP =  0, TOP_K = 10 | EM = 16.7%, F1 = 33.3%
CHUNK_WORDS = 150, OVERLAP = 50, TOP_K =  3 | EM = 8.3%, F1 = 25.0%
CHUNK_WORDS = 150, OVERLAP = 50, TOP_K =  5 | EM = 16.7%, F1 = 27.8%
CHUNK_WORDS = 150, OVERLAP = 50, TOP_K = 10 | EM = 16.7%, F1 = 27.8%
CHUNK_WORDS = 200, OVERLAP =  0, TOP_K =  3 | EM = 8.3%, F1 = 25.0%
CHUNK_WORDS = 200, OVERLAP =  0, TOP_K =  5 | EM = 8.3%, F1 = 19.4%
CHUNK_WORDS = 200, OVERLAP =  0, TOP_K = 1

In [29]:
## 6.3. Selecting the best setting (hightest F1, then highest EM) and rebuilding with it
best = tuning_results[0]
for result in tuning_results:
    chunk_words, overlap, top_k, em, f1 = result
    if f1 > best[4] or (f1 == best[4] and em > best[3]):
        best = result
    
CHUNK_WORDS, OVERLAP, TOP_K, best_em, best_f1 = best
RebuildingPassages(CHUNK_WORDS, OVERLAP) # chatbot using best passages

print('Best setting: CHUNK_WORDS = {}, OVERLAP = {}, TOP_K = {}'.format(CHUNK_WORDS, OVERLAP, TOP_K))
print('Exact Match: {:.1%} | F1 Score: {:.1%}'.format(best_em, best_f1))
print()

# 6.3.1. Per-question results with the best setting
average_em, average_f1 = EvaluateChatbot(TOP_K, show_details = True)

Best setting: CHUNK_WORDS = 100, OVERLAP = 50, TOP_K = 5
Exact Match: 16.7% | F1 Score: 34.4%

EM = 0, F1 = 0.00 | Q: Where did Victor go to university?
 Predicted: Geneva | Correct: Ingolstadt
EM = 0, F1 = 0.80 | Q: To whom does Robert Walton write his letters?
 Predicted: Mrs. Saville, England | Correct: Mrs. Saville
EM = 1, F1 = 1.00 | Q: What city is Victor's family from?
 Predicted: Geneva | Correct: Geneva
EM = 0, F1 = 0.67 | Q: Which professor inspired Victor to study chemistry?
 Predicted: M. Waldman | Correct: Waldman
EM = 0, F1 = 0.67 | Q: Which professor dismissed the alchemists Victor had read?
 Predicted: M. Krempe | Correct: Krempe
EM = 0, F1 = 0.00 | Q: Which alchemist did Victor read as a boy?
 Predicted: M. Krempe | Correct: Cornelius Agrippa
EM = 0, F1 = 0.00 | Q: Who is Victor's closest friend?
 Predicted: Louisa Biron | Correct: Clerval
EM = 0, F1 = 0.00 | Q: What is the name of Victor's youngest brother?
 Predicted: Manon | Correct: William
EM = 1, F1 = 1.00 | Q: W